Think of query optimization as:

User Query -> Transform Query -> Better Retrieval

Approaches to do it are -
1. Query Re-Writing
2. Multi Query Generation
3. Query Expansion
4. HyDE
5. Step Back Query
6. Query Decomposition
7. Intent Classification
8. MetaData Extraction
9. Query Routing
10. KeyWords Extraction
11. Semantic Compression
12. Contextual Enrichment
13. Re-Ranking

NOTE: HyDE and Re-Ranking are not usually done using prompts in production-grade RAG.

#PROMPT TEMPLATES

| Technique             | What It Does                       | Production Usually Uses       |
| --------------------- | ---------------------------------- | ----------------------------- |
| Query Rewriting       | clarify query                      | LLM/small rewrite models      |
| Multi Query           | multiple semantic searches         | LLM + retrieval orchestration |
| Query Expansion       | add related terms                  | NLP/IR systems                |
| HyDE                  | generate fake answer for embedding | LLM + optimized retrieval     |
| Step-Back Query       | broaden abstraction                | LLM sometimes                 |
| Query Decomposition   | split complex query                | planners/agents               |
| Intent Classification | choose pipeline                    | classifiers/models            |
| Metadata Extraction   | extract filters/entities           | parsers/NER/rules             |
| Query Routing         | choose retriever/tool              | orchestration systems         |
| Keyword Extraction    | important retrieval terms          | NLP/search algorithms         |
| Semantic Compression  | shorten semantic meaning           | compression/retrieval systems |
| Contextual Enrichment | add memory/session context         | memory/profile systems        |
| Reranking             | reorder retrieved chunks           | cross-encoders                |


**Query Re-Writing**

Query rewriting converts a vague, conversational, or incomplete user query into a clearer and retrieval-optimized standalone query. It helps the retriever better understand the actual intent of the user, especially in conversational RAG systems where users ask follow-up questions like “what about pricing?” instead of fully specifying the topic again.

In [ ]:
query_rewrite_prompt = f"""
You are an expert retrieval query rewriting system.

Your task is to rewrite the user's query into a
clear, standalone, retrieval-optimized query.

Guidelines:
- Preserve original meaning
- Resolve vague references
- Include important entities
- Remove conversational fluff
- Make the query retrieval-friendly
- Keep it concise
- Do NOT answer the question

Previous Conversation:
{chat_context}

Current User Query:
{query}

Return ONLY the rewritten query.
"""

**Multi Query Generation**

Multi query generation creates several semantically different versions of the same query so the system can retrieve documents from multiple perspectives. Instead of relying on one embedding search, the retriever searches using multiple related phrasings, improving recall and reducing the chance of missing relevant documents.

In [ ]:
multi_query_prompt = f"""
You are an advanced retrieval query generation system.

Generate multiple diverse search queries that
capture different semantic perspectives of the
user's question.

Guidelines:
- Each query should explore a different angle
- Use different terminology and phrasing
- Preserve relevance to the original query
- Keep queries concise
- Avoid duplicate meaning
- Do NOT answer the question

User Query:
{query}

Return exactly 4 queries.

One query per line.
"""

**Query Expansion**

Query expansion adds related keywords, synonyms, technical terms, and associated concepts to the original query. Its purpose is to improve retrieval coverage by including terms that may appear in documents even if the user did not explicitly mention them.

In [ ]:
query_expansion_prompt = f"""
You are a retrieval query expansion system.

Expand the user's query with:
- related keywords
- synonyms
- associated technical terms
- relevant concepts
- domain-specific terminology

Guidelines:
- Preserve original intent
- Add semantically related terms
- Improve retrieval coverage
- Keep expansion concise
- Do NOT answer the question

User Query:
{query}

Return ONLY the expanded query.
"""

**Step-Back Query**

Step-back querying transforms a narrow or highly specific query into a broader conceptual version. This helps retrieve documents that discuss the general concept behind the question, which is useful because many documents are written in broader explanatory language rather than exact user phrasing.

In [ ]:
#5. Step Back Query
step_back_prompt = f"""
You are a retrieval query abstraction system.

Generate a broader conceptual version of the
user's query to improve document retrieval.

Guidelines:
- Capture the underlying concept
- Generalize the query slightly
- Preserve original intent
- Make retrieval easier
- Do NOT answer the question
- Keep concise

User Query:
{query}

Return ONLY the broader query.
"""

**Query Decomposition**

Query decomposition breaks a complex or multi-part question into smaller independent sub-queries. Each sub-query is retrieved separately, allowing the system to gather focused information for each aspect of the original question before combining the results into a final answer.

In [ ]:
#6. Query Decomposition
query_decomposition_prompt = f"""
You are a query decomposition system.

Break the user's query into smaller,
independent retrieval-focused sub-queries.

Guidelines:
- Each sub-query should represent one concept
- Preserve original meaning
- Make sub-queries retrieval-friendly
- Avoid overlap between sub-queries
- Do NOT answer the question

User Query:
{query}

Return one sub-query per line.
"""

**Intent Classification**

Intent classification identifies the type of user request, such as retrieval, summarization, troubleshooting, comparison, or casual conversation. The detected intent determines which retrieval pipeline, tools, prompts, or workflows should be used to handle the query effectively.

In [ ]:
#7. Intent Classfication and Query Rerouting
intent_classification_prompt = f"""
You are an intent classification system.

Classify the user query into ONE category.

Available Categories:
- retrieval
- summarization
- comparison
- troubleshooting
- explanation
- chit_chat

Guidelines:
- Choose ONLY one category
- Return ONLY category name
- No explanation

User Query:
{query}
"""

**Metadata Extraction**

Metadata extraction identifies structured information from the query, such as company names, dates, products, document types, or regions. These extracted filters help narrow retrieval results and improve precision by searching only within relevant subsets of the data.

In [ ]:
#8. MetaData Extraction
metadata_extraction_prompt = f"""
You are a metadata extraction system.

Extract structured metadata filters from the query.

Possible fields:
- topic
- product
- company
- year
- document_type
- region
- technology

Guidelines:
- Return JSON only
- Use null if unavailable
- Do NOT explain anything

User Query:
{query}
"""

**Keyword Extraction**

Keyword extraction identifies the most important terms and entities in the query for retrieval purposes. These keywords can improve traditional search systems like BM25 and help retrieval systems focus on the core concepts of the user’s question.

In [ ]:
#10. Keywords Extraction
keyword_extraction_prompt = f"""
You are a keyword extraction system.

Extract important retrieval-focused keywords
from the user's query.

Guidelines:
- Extract technical terms
- Extract entities
- Extract products/services
- Extract important concepts
- Remove filler words
- Keep keywords concise
- Do NOT answer the question

User Query:
{query}

Return keywords separated by commas.
"""

**Semantic Compression**

Semantic compression reduces unnecessary words while preserving the core meaning of a query or retrieved context. It is commonly used to optimize token usage, reduce noise, and make retrieval or prompting more efficient without losing important information.

In [ ]:
#11. Semantic Compression
semantic_compression_prompt = f"""
You are a semantic compression system.

Compress the user's query into a concise,
retrieval-optimized semantic query.

Guidelines:
- Preserve meaning
- Remove conversational fluff
- Remove unnecessary words
- Keep core semantic intent
- Make retrieval-friendly
- Do NOT answer the question

User Query:
{query}

Return ONLY the compressed query.
"""

**Contextual Enrichment**

Contextual enrichment enhances the query using conversation history, memory, user context, or application state. It resolves ambiguous references and adds missing context so the retrieval system understands exactly what the user is referring to.

In [ ]:
#12. Contextual Enrichment
contextual_enrichment_prompt = f"""
You are a contextual query enrichment system.

Use the previous conversation to enrich
the current query for better retrieval.

Guidelines:
- Resolve ambiguous references
- Include missing entities/topics
- Preserve original intent
- Use relevant conversation context
- Improve retrieval quality
- Do NOT answer the question

Previous Conversation:
{chat_context}

Current User Query:
{query}

Return ONLY the enriched query.
"""

#Step 1: Install Libraries

In [ ]:
!pip install -q \
supabase \
sentence-transformers \
groq \
pypdf \
langchain \
tiktoken

In [ ]:
import os
import uuid
import numpy as np

from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
from langchain.text_splitter import RecursiveCharacterTextSplitter
from supabase import create_client
from groq import Groq

# Step 2: Supabase and LLM Setup

In [ ]:
SUPABASE_URL = "YOUR_SUPABASE_URL"

SUPABASE_KEY = "YOUR_SUPABASE_SERVICE_ROLE_KEY"

supabase = create_client(
    SUPABASE_URL,
    SUPABASE_KEY
)

In [ ]:
GROQ_API_KEY = "YOUR_GROQ_API_KEY"

client = Groq(
    api_key=GROQ_API_KEY
)

#Step 3: Load Embedding Model

In [ ]:
model = SentenceTransformer(
    "BAAI/bge-small-en-v1.5"
)

In [ ]:
splitter = RecursiveCharacterTextSplitter(

    chunk_size=1000,

    chunk_overlap=200
)

#Step 4: PDF Text Extraction

In [ ]:
def extract_text_from_pdf(file_path):

    reader = PdfReader(file_path)

    pages = []

    for i, page in enumerate(reader.pages):

        text = page.extract_text()

        if text:

            pages.append({

                "page_number": i + 1,

                "text": text
            })

    return pages

In [ ]:
all_rows = []

for file_name in os.listdir(PDF_FOLDER):

    if file_name.endswith(".pdf"):

        print(f"Processing: {file_name}")

        file_path = os.path.join(
            PDF_FOLDER,
            file_name
        )

        # ====================================================
        # EXTRACT PDF TEXT
        # ====================================================

        pages = extract_text_from_pdf(
            file_path
        )

        # ====================================================
        # CHUNK EACH PAGE
        # ====================================================

        for page in pages:

            chunks = splitter.split_text(
                page["text"]
            )

            for chunk_index, chunk in enumerate(chunks):

                all_rows.append({

                    "id": str(uuid.uuid4()),

                    "file_name":
                    file_name,

                    "page_number":
                    page["page_number"],

                    "chunk_index":
                    chunk_index,

                    "chunk_text":
                    chunk
                })

print(
    f"\nTotal Chunks: {len(all_rows)}"
)


#Sep 5: Create Embeddings

In [ ]:
texts = [

    row["chunk_text"]

    for row in all_rows
]

embeddings = model.encode(

    texts,

    normalize_embeddings=True,

    show_progress_bar=True
)

#Step 6: Supabase Storage

In [ ]:
insert_rows = []

for i, row in enumerate(all_rows):

    insert_rows.append({

        "id":
        row["id"],

        "file_name":
        row["file_name"],

        "page_number":
        row["page_number"],

        "chunk_index":
        row["chunk_index"],

        "chunk_text":
        row["chunk_text"],

        "embedding":
        embeddings[i].tolist()
    })

# ============================================================
# STEP 12 — INSERT INTO SUPABASE
# ============================================================

response = supabase.table(
    "document_chunks"
).insert(
    insert_rows
).execute()

print("\nInserted Into Supabase")

#Step 7: Chat Loop

In [ ]:
SESSION_ID = "session_1"

while True:

    # ========================================================
    # USER QUERY
    # ========================================================

    query = input("\nAsk Question: ")

    if query.lower() == "exit":
        break

    # ========================================================
    # OPTIONAL QUERY TRANSFORMATION
    # ========================================================

    transformed_query = query

    # ========================================================
    # CREATE QUERY EMBEDDING
    # ========================================================

    query_embedding = model.encode(

        transformed_query,

        normalize_embeddings=True
    ).tolist()

    # ========================================================
    # VECTOR RETRIEVAL
    # ========================================================

    retrieval = supabase.rpc(

        "match_documents",

        {

            "query_embedding":
            query_embedding,

            "match_count":
            5
        }

    ).execute()

    # ========================================================
    # BUILD DOCUMENT CONTEXT
    # ========================================================

    document_context = ""

    for row in retrieval.data:

        document_context += (

            f"\n[FILE: {row['file_name']}]\n"

            f"{row['chunk_text']}\n"
        )

    # ========================================================
    # PROMPT TEMPLATE
    # ========================================================

    prompt = f"""

    =======================================================
    SYSTEM / PROMPT TEMPLATE SPACE
    =======================================================

    ADD:
    - SYSTEM PROMPT
    - MEMORY
    - QUERY REWRITING
    - HYDE
    - MULTI QUERY
    - AGENTIC FLOW
    - REASONING
    - TOOL CALLS
    - OUTPUT FORMAT

    =======================================================
    RETRIEVED DOCUMENTS
    =======================================================

    {document_context}

    =======================================================
    USER QUESTION
    =======================================================

    {query}

    =======================================================
    ANSWER
    =======================================================

    """

    # ========================================================
    # LLM CALL
    # ========================================================

    response = client.chat.completions.create(

        model="llama-3.3-70b-versatile",

        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    answer = (
        response
        .choices[0]
        .message
        .content
    )

    # ========================================================
    # PRINT ANSWER
    # ========================================================

    print("\n===================================")
    print("ANSWER")
    print("===================================\n")

    print(answer)